
# Collect all OpenAI, Claude and Gemini responses

## What this notebook does

This notebook reads the workbook located at the **exact Google Drive path**:

```text
/content/drive/MyDrive/pre_data_collection.xlsx
```

It then fills the existing response fields for all 1,000 prompts across:

- Technology
- Finance
- Healthcare
- Marketing
- Sports

For every prompt it collects:

- one OpenAI response;
- one Claude response;
- one Gemini response;
- exact model used;
- UTC timestamp;
- temperature setting;
- input-token count;
- output-token count.

The completed workbook is saved as:

```text
/content/drive/MyDrive/raw_dataset.xlsx
```

A persistent checkpoint is saved as:

```text
/content/drive/MyDrive/data_collection_all_responses_checkpoint.jsonl
```

## Safety and reliability

- The original `pre_data_collection.xlsx` is never overwritten.
- Existing completed response cells are skipped.
- Every successful paid API response is written to the checkpoint immediately.
- The workbook is copied back to Google Drive regularly.
- If Colab disconnects, rerun the notebook and it resumes.
- If a model is retired, unavailable, overloaded or rate-limited, the notebook
  tries another accessible model from the **same provider**.
- Authentication, billing and exhausted-quota errors disable only the affected
  provider for the current run rather than repeatedly spending time retrying.

## Required Colab Secrets

Open the key icon in Colab and enable notebook access for:

- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GEMINI_API_KEY`

## Recommended workflow

1. Run cells 1–6 in order.
2. Run the one-row test.
3. Inspect the three saved responses.
5. Run the full-collection cell.
6. Keep the Colab tab open while collection is running.
7. If the session disconnects, reconnect and run all cells again. Completed
   responses will be skipped.


## 1. Install the required libraries

In [ ]:

!pip -q install -U openai anthropic google-genai openpyxl


## 2. Mount Google Drive and confirm the exact workbook

In [ ]:

import os
import json
import shutil
import time
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive, files, userdata
from openpyxl import load_workbook


warnings.filterwarnings(
    "ignore",
    message="Unknown extension is not supported and will be removed",
)
warnings.filterwarnings(
    "ignore",
    message="Conditional Formatting extension is not supported and will be removed",
)


drive.mount("/content/drive")

INPUT_FILE = Path(
    "/content/drive/pre_data_collection.xlsx"
)
OUTPUT_FILE = Path(
    "/content/drive/MyDrive/data_collection_all_responses.xlsx"
)
CHECKPOINT_FILE = Path(
    "/content/drive/MyDrive/data_collection_all_responses_checkpoint.jsonl"
)
LOCAL_WORKING_FILE = Path(
    "/content/data_collection_all_responses_working.xlsx"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "The required workbook was not found at:\n"
        f"{INPUT_FILE}\n\n"
        "Confirm that the Google Drive file is named exactly "
        "'pre_data_collection.xlsx' and is located directly in My Drive."
    )

print(f"Confirmed input workbook: {INPUT_FILE}")
print(f"Input workbook size: {INPUT_FILE.stat().st_size:,} bytes")

# Resume a previous run when an output already exists.
if OUTPUT_FILE.exists():
    shutil.copy2(OUTPUT_FILE, LOCAL_WORKING_FILE)
    print(f"Resuming existing output: {OUTPUT_FILE}")
else:
    shutil.copy2(INPUT_FILE, LOCAL_WORKING_FILE)
    shutil.copy2(LOCAL_WORKING_FILE, OUTPUT_FILE)
    print(f"Created new output copy: {OUTPUT_FILE}")

print(f"Persistent checkpoint: {CHECKPOINT_FILE}")


## 3. Validate the workbook structure

In [ ]:

DOMAIN_SHEETS = [
    "Technology",
    "Finance",
    "Healthcare",
    "Marketing",
    "Sports",
]

EXPECTED_HEADERS = {
    1: "Prompt ID",
    7: "Prompt",
    12: "ChatGPT Model Version",
    13: "ChatGPT Timestamp",
    14: "ChatGPT Temperature",
    15: "ChatGPT Input Tokens",
    16: "ChatGPT Output Tokens",
    17: "ChatGPT Response Text",
    25: "Claude Model Version",
    26: "Claude Timestamp",
    27: "Claude Temperature",
    28: "Claude Input Tokens",
    29: "Claude Output Tokens",
    30: "Claude Response Text",
    38: "Gemini Model Version",
    39: "Gemini Timestamp",
    40: "Gemini Temperature",
    41: "Gemini Input Tokens",
    42: "Gemini Output Tokens",
    43: "Gemini Response Text",
    51: "Completion Status",
    54: "Data Collection Notes",
}


def validate_workbook(path):
    workbook = load_workbook(
        path,
        read_only=True,
        data_only=False,
    )

    try:
        for sheet_name in DOMAIN_SHEETS:
            if sheet_name not in workbook.sheetnames:
                raise ValueError(
                    f"Required sheet is missing: {sheet_name}"
                )

            worksheet = workbook[sheet_name]

            for column_number, expected_header in EXPECTED_HEADERS.items():
                actual_header = worksheet.cell(
                    row=1,
                    column=column_number,
                ).value

                if actual_header != expected_header:
                    raise ValueError(
                        f"{sheet_name}: expected '{expected_header}' "
                        f"in column {column_number}, but found "
                        f"'{actual_header}'."
                    )

            prompt_count = sum(
                1
                for row_number in range(2, worksheet.max_row + 1)
                if worksheet.cell(
                    row=row_number,
                    column=7,
                ).value
            )

            print(
                f"{sheet_name}: {prompt_count} prompts; "
                "structure confirmed."
            )

    finally:
        workbook.close()


validate_workbook(LOCAL_WORKING_FILE)
print("\nWorkbook validation passed.")


## 4. Load API keys and create official SDK clients

In [ ]:

from openai import OpenAI
from anthropic import Anthropic
from google import genai
from google.genai import types


def get_secret(*possible_names):
    for name in possible_names:
        try:
            value = userdata.get(name)

            if value:
                print(f"Loaded secret: {name}")
                return value

        except Exception:
            pass

    raise ValueError(
        "No usable Colab secret was found. Tried: "
        + ", ".join(possible_names)
        + ". Check the secret name and enable notebook access."
    )


OPENAI_API_KEY = get_secret(
    "OPENAI_API_KEY",
    "OPENAI_API",
    "OPENAI_A",
)

ANTHROPIC_API_KEY = get_secret(
    "ANTHROPIC_API_KEY",
    "CLAUDE_API_KEY",
    "ANTHROPIC_API",
    "ANTHROPE",
)

GEMINI_API_KEY = get_secret(
    "GEMINI_API_KEY",
    "GOOGLE_API_KEY",
    "GEMINI_API",
    "GEMINI_A",
)


openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    timeout=180.0,
    max_retries=0,
)

claude_client = Anthropic(
    api_key=ANTHROPIC_API_KEY,
    timeout=180.0,
    max_retries=0,
)

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY,
)

print("\nAll three API clients were created successfully.")



## 5. Discover models available to your API keys

This cell calls only model-listing endpoints. It does not generate responses.

The notebook prioritises smaller and generally lower-cost text models, then
keeps additional accessible models as fallbacks. The exact successful model
is written into the workbook for every response.


In [ ]:

MAX_MODELS_PER_PROVIDER = 10

OPENAI_PREFERRED_IDS = [
    "gpt-5-mini",
    "gpt-4.1-mini",
    "gpt-4o-mini",
]

CLAUDE_PREFERRED_FAMILIES = [
    "haiku",
    "sonnet",
    "opus",
]

GEMINI_PREFERRED_IDS = [
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-flash-latest",
]


def unique_in_order(values):
    output = []
    seen = set()

    for value in values:
        value = str(value).strip()

        if value and value not in seen:
            seen.add(value)
            output.append(value)

    return output


def strip_models_prefix(value):
    value = str(value or "")

    if value.startswith("models/"):
        return value.split("/", 1)[1]

    return value


def prioritise_exact_and_families(
    available,
    exact_ids,
    family_terms,
):
    ordered = []

    for model_id in exact_ids:
        if model_id in available:
            ordered.append(model_id)

    for term in family_terms:
        for model_id in available:
            if (
                term in model_id.lower()
                and model_id not in ordered
            ):
                ordered.append(model_id)

    for model_id in available:
        if model_id not in ordered:
            ordered.append(model_id)

    return ordered[:MAX_MODELS_PER_PROVIDER]


def discover_openai_models():
    excluded_terms = (
        "audio",
        "image",
        "realtime",
        "transcribe",
        "tts",
        "moderation",
        "embedding",
        "search",
        "codex",
        "deep-research",
        "computer-use",
    )

    page = openai_client.models.list()

    available = sorted(
        {
            str(model.id)
            for model in page.data
            if getattr(model, "id", None)
            and str(model.id).startswith("gpt-")
            and not any(
                term in str(model.id).lower()
                for term in excluded_terms
            )
        }
    )

    return prioritise_exact_and_families(
        available,
        OPENAI_PREFERRED_IDS,
        (
            "mini",
            "gpt-5",
            "gpt-4.1",
            "gpt-4o",
        ),
    )


def discover_claude_models():
    page = claude_client.models.list(limit=100)
    data = getattr(page, "data", page)

    available = sorted(
        {
            str(model.id)
            for model in data
            if getattr(model, "id", None)
            and str(model.id).startswith("claude-")
        }
    )

    return prioritise_exact_and_families(
        available,
        (),
        CLAUDE_PREFERRED_FAMILIES,
    )


def discover_gemini_models():
    excluded_terms = (
        "gemini-2.0",
        "embedding",
        "imagen",
        "image",
        "veo",
        "tts",
        "speech",
        "live",
        "aqa",
    )

    available = []

    for model in gemini_client.models.list():
        model_id = strip_models_prefix(
            getattr(model, "name", "")
        )

        if not model_id.startswith("gemini-"):
            continue

        if any(
            term in model_id.lower()
            for term in excluded_terms
        ):
            continue

        supported_actions = (
            getattr(model, "supported_actions", None)
            or getattr(
                model,
                "supported_generation_methods",
                None,
            )
            or []
        )

        actions_text = " ".join(
            str(action).lower().replace("_", "")
            for action in supported_actions
        )

        if (
            supported_actions
            and "generatecontent" not in actions_text
        ):
            continue

        available.append(model_id)

    available = sorted(set(available))

    return prioritise_exact_and_families(
        available,
        GEMINI_PREFERRED_IDS,
        (
            "flash-lite",
            "flash",
            "pro",
        ),
    )


MODEL_POOLS = {
    "ChatGPT": discover_openai_models(),
    "Claude": discover_claude_models(),
    "Gemini": discover_gemini_models(),
}

for provider, models in MODEL_POOLS.items():
    if not models:
        raise RuntimeError(
            f"No usable text models were found for {provider}. "
            "Check the API key, account access and billing."
        )

    print(f"\n{provider} fallback order:")

    for index, model_id in enumerate(models, start=1):
        print(f"  {index}. {model_id}")

print(
    "\nModel discovery finished. "
    "No prompt responses were generated in this cell."
)


## 6. Define API calls, fallback handling and workbook saving

In [ ]:

MAX_OUTPUT_TOKENS = 1200
MAX_TEMPORARY_ROUNDS = 2
WAIT_BETWEEN_TEMPORARY_ROUNDS_SECONDS = 20
PAUSE_AFTER_SUCCESS_SECONDS = 0.5
SAVE_WORKBOOK_EVERY_SUCCESSES = 10

PROVIDER_COLUMNS = {
    "ChatGPT": {
        "model": 12,
        "timestamp": 13,
        "temperature": 14,
        "input_tokens": 15,
        "output_tokens": 16,
        "response_text": 17,
    },
    "Claude": {
        "model": 25,
        "timestamp": 26,
        "temperature": 27,
        "input_tokens": 28,
        "output_tokens": 29,
        "response_text": 30,
    },
    "Gemini": {
        "model": 38,
        "timestamp": 39,
        "temperature": 40,
        "input_tokens": 41,
        "output_tokens": 42,
        "response_text": 43,
    },
}

PROMPT_ID_COLUMN = 1
PROMPT_COLUMN = 7
COMPLETION_STATUS_COLUMN = 51
DATA_COLLECTION_NOTES_COLUMN = 54

ACTIVE_MODEL = {
    provider: models[0]
    for provider, models in MODEL_POOLS.items()
}

PERMANENTLY_BAD_MODELS = {
    provider: set()
    for provider in MODEL_POOLS
}

PROVIDER_DISABLED_REASON = {}
MODEL_USAGE = Counter()


class ProviderDisabledError(RuntimeError):
    pass


def utc_timestamp():
    return datetime.now(timezone.utc).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )


def get_status_code(exception):
    possible_values = [
        getattr(exception, "status_code", None),
        getattr(exception, "code", None),
        getattr(
            getattr(exception, "response", None),
            "status_code",
            None,
        ),
    ]

    for value in possible_values:
        try:
            if value is not None:
                return int(value)
        except Exception:
            pass

    return None


def exception_text(exception):
    return str(exception).lower()


def is_provider_fatal(exception):
    code = get_status_code(exception)
    text = exception_text(exception)

    if code in (401, 402):
        return True

    phrases = (
        "invalid api key",
        "incorrect api key",
        "authentication_error",
        "payment required",
        "insufficient_quota",
        "exceeded your current quota",
        "credit balance",
        "billing hard limit",
        "account is not active",
    )

    return any(
        phrase in text
        for phrase in phrases
    )


def is_permanent_model_failure(exception):
    code = get_status_code(exception)
    text = exception_text(exception)

    if code in (400, 403, 404, 405, 422):
        return True

    phrases = (
        "model not found",
        "does not exist",
        "not have access",
        "no longer available",
        "deprecated",
        "retired",
        "unsupported model",
        "invalid model",
        "not supported for this model",
        "not supported for generatecontent",
        "not supported for generate_content",
    )

    return any(
        phrase in text
        for phrase in phrases
    )


def is_temporary_failure(exception):
    code = get_status_code(exception)
    text = exception_text(exception)

    if code in (
        408,
        409,
        429,
        500,
        502,
        503,
        504,
        529,
    ):
        return True

    phrases = (
        "overload",
        "overloaded",
        "high demand",
        "unavailable",
        "capacity",
        "temporarily",
        "try again later",
        "rate limit",
        "too many requests",
        "timeout",
        "timed out",
        "connection reset",
        "connection error",
    )

    return any(
        phrase in text
        for phrase in phrases
    )


def extract_openai_text(response):
    text = getattr(response, "output_text", None)

    if text:
        return str(text).strip()

    pieces = []

    for item in getattr(response, "output", []) or []:
        for content in getattr(item, "content", []) or []:
            if getattr(content, "type", None) == "output_text":
                value = getattr(content, "text", None)

                if value:
                    pieces.append(str(value))

    return "\n".join(pieces).strip()


def extract_claude_text(message):
    pieces = []

    for block in getattr(message, "content", []) or []:
        if getattr(block, "type", None) == "text":
            value = getattr(block, "text", None)

            if value:
                pieces.append(str(value))

    return "\n".join(pieces).strip()


def extract_gemini_text(response):
    try:
        text = response.text

        if text:
            return str(text).strip()
    except Exception:
        pass

    pieces = []

    for candidate in getattr(response, "candidates", []) or []:
        content = getattr(candidate, "content", None)

        for part in getattr(content, "parts", []) or []:
            value = getattr(part, "text", None)

            if value:
                pieces.append(str(value))

    return "\n".join(pieces).strip()


def call_openai(model_id, prompt):
    response = openai_client.responses.create(
        model=model_id,
        input=prompt,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )

    usage = getattr(response, "usage", None)

    return {
        "model": getattr(response, "model", None) or model_id,
        "timestamp": utc_timestamp(),
        "temperature": "provider default",
        "input_tokens": getattr(usage, "input_tokens", None),
        "output_tokens": getattr(usage, "output_tokens", None),
        "response_text": extract_openai_text(response),
    }


def call_claude(model_id, prompt):
    message = claude_client.messages.create(
        model=model_id,
        max_tokens=MAX_OUTPUT_TOKENS,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    usage = getattr(message, "usage", None)

    return {
        "model": getattr(message, "model", None) or model_id,
        "timestamp": utc_timestamp(),
        "temperature": "provider default",
        "input_tokens": getattr(usage, "input_tokens", None),
        "output_tokens": getattr(usage, "output_tokens", None),
        "response_text": extract_claude_text(message),
    }


def call_gemini(model_id, prompt):
    response = gemini_client.models.generate_content(
        model=model_id,
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=MAX_OUTPUT_TOKENS,
        ),
    )

    usage = getattr(response, "usage_metadata", None)

    return {
        "model": (
            getattr(response, "model_version", None)
            or model_id
        ),
        "timestamp": utc_timestamp(),
        "temperature": "provider default",
        "input_tokens": getattr(
            usage,
            "prompt_token_count",
            None,
        ),
        "output_tokens": getattr(
            usage,
            "candidates_token_count",
            None,
        ),
        "response_text": extract_gemini_text(response),
    }


PROVIDER_CALLS = {
    "ChatGPT": call_openai,
    "Claude": call_claude,
    "Gemini": call_gemini,
}


def ordered_models(provider):
    active = ACTIVE_MODEL.get(provider)

    return unique_in_order(
        ([active] if active else [])
        + MODEL_POOLS[provider]
    )


def call_with_fallback(provider, prompt):
    if provider in PROVIDER_DISABLED_REASON:
        raise ProviderDisabledError(
            PROVIDER_DISABLED_REASON[provider]
        )

    attempt_log = []
    last_exception = None

    for round_number in range(
        1,
        MAX_TEMPORARY_ROUNDS + 1,
    ):
        temporary_failure_seen = False
        attempted_model_count = 0

        for model_id in ordered_models(provider):
            if model_id in PERMANENTLY_BAD_MODELS[provider]:
                continue

            attempted_model_count += 1

            try:
                print(
                    f"    Trying {model_id} "
                    f"(round {round_number})"
                )

                result = PROVIDER_CALLS[provider](
                    model_id,
                    prompt,
                )

                if not result.get("response_text"):
                    raise RuntimeError(
                        "The API returned no usable response text."
                    )

                ACTIVE_MODEL[provider] = model_id
                MODEL_USAGE[
                    f"{provider}|{result['model']}"
                ] += 1

                return result, attempt_log

            except KeyboardInterrupt:
                raise

            except Exception as exc:
                last_exception = exc
                code = get_status_code(exc)

                attempt_log.append(
                    {
                        "model": model_id,
                        "status_code": code,
                        "error": str(exc)[:700],
                    }
                )

                if is_provider_fatal(exc):
                    PROVIDER_DISABLED_REASON[provider] = str(exc)

                    raise ProviderDisabledError(
                        str(exc)
                    ) from exc

                if is_permanent_model_failure(exc):
                    PERMANENTLY_BAD_MODELS[provider].add(
                        model_id
                    )

                    print(
                        f"    {model_id} cannot be used "
                        f"({code or type(exc).__name__}); "
                        "trying the next model."
                    )
                    continue

                if is_temporary_failure(exc):
                    temporary_failure_seen = True

                    print(
                        f"    {model_id} is temporarily unavailable "
                        f"({code or type(exc).__name__}); "
                        "trying the next model."
                    )
                    continue

                raise

        if (
            attempted_model_count > 0
            and temporary_failure_seen
            and round_number < MAX_TEMPORARY_ROUNDS
        ):
            print(
                "    All usable models had temporary failures. "
                f"Waiting {WAIT_BETWEEN_TEMPORARY_ROUNDS_SECONDS} "
                "seconds before one more round."
            )

            time.sleep(
                WAIT_BETWEEN_TEMPORARY_ROUNDS_SECONDS
            )
            continue

        break

    summary = " | ".join(
        f"{entry['model']} [{entry['status_code']}]"
        for entry in attempt_log
    )

    raise RuntimeError(
        f"All fallback models failed for {provider}. "
        f"{summary}. Last error: {last_exception}"
    )


def checkpoint_key(sheet_name, row_number, provider):
    return f"{sheet_name}|{row_number}|{provider}"


def load_checkpoint():
    records = {}

    if not CHECKPOINT_FILE.exists():
        return records

    with CHECKPOINT_FILE.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

                records[
                    checkpoint_key(
                        record["sheet"],
                        int(record["row"]),
                        record["provider"],
                    )
                ] = record

            except Exception as exc:
                print(
                    f"Skipping damaged checkpoint line "
                    f"{line_number}: {exc}"
                )

    return records


def append_checkpoint(record):
    with CHECKPOINT_FILE.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )
        file.flush()
        os.fsync(file.fileno())


def write_result(
    worksheet,
    row_number,
    provider,
    result,
):
    columns = PROVIDER_COLUMNS[provider]

    for field, column_number in columns.items():
        worksheet.cell(
            row=row_number,
            column=column_number,
        ).value = result.get(field)


def append_note(
    worksheet,
    row_number,
    note,
):
    cell = worksheet.cell(
        row=row_number,
        column=DATA_COLLECTION_NOTES_COLUMN,
    )

    existing = (
        str(cell.value).strip()
        if cell.value
        else ""
    )

    cell.value = (
        f"{existing}\n{note}"
        if existing
        else note
    )


def update_completion_status(
    worksheet,
    row_number,
):
    response_values = [
        worksheet.cell(
            row=row_number,
            column=PROVIDER_COLUMNS[
                provider
            ]["response_text"],
        ).value
        for provider in (
            "ChatGPT",
            "Claude",
            "Gemini",
        )
    ]

    present = [
        value is not None
        and str(value).strip() != ""
        for value in response_values
    ]

    if all(present):
        status = "Complete"
    elif any(present):
        status = "In progress"
    else:
        status = "Not started"

    worksheet.cell(
        row=row_number,
        column=COMPLETION_STATUS_COLUMN,
    ).value = status


def save_workbook_to_drive(workbook):
    temporary_file = Path(
        "/content/data_collection_save_temporary.xlsx"
    )

    try:
        workbook.calculation.calcMode = "auto"
        workbook.calculation.fullCalcOnLoad = True
        workbook.calculation.forceFullCalc = True
    except Exception:
        pass

    workbook.save(temporary_file)
    shutil.copy2(temporary_file, LOCAL_WORKING_FILE)
    shutil.copy2(temporary_file, OUTPUT_FILE)

    if temporary_file.exists():
        temporary_file.unlink()


def restore_checkpoint(workbook):
    restored = 0

    for record in load_checkpoint().values():
        sheet_name = record.get("sheet")
        row_number = int(record.get("row"))
        provider = record.get("provider")
        result = record.get("result") or {}

        if (
            sheet_name not in workbook.sheetnames
            or provider not in PROVIDER_COLUMNS
            or not result.get("response_text")
        ):
            continue

        worksheet = workbook[sheet_name]

        current_prompt_id = worksheet.cell(
            row=row_number,
            column=PROMPT_ID_COLUMN,
        ).value

        if current_prompt_id != record.get("prompt_id"):
            continue

        response_column = PROVIDER_COLUMNS[
            provider
        ]["response_text"]

        existing = worksheet.cell(
            row=row_number,
            column=response_column,
        ).value

        if existing is None or not str(existing).strip():
            write_result(
                worksheet,
                row_number,
                provider,
                result,
            )

            update_completion_status(
                worksheet,
                row_number,
            )

            restored += 1

    return restored


def calculate_progress(workbook):
    saved = Counter()
    missing = Counter()

    for sheet_name in DOMAIN_SHEETS:
        worksheet = workbook[sheet_name]

        for row_number in range(
            2,
            worksheet.max_row + 1,
        ):
            prompt = worksheet.cell(
                row=row_number,
                column=PROMPT_COLUMN,
            ).value

            if prompt is None or not str(prompt).strip():
                continue

            for provider in (
                "ChatGPT",
                "Claude",
                "Gemini",
            ):
                response_column = PROVIDER_COLUMNS[
                    provider
                ]["response_text"]

                response = worksheet.cell(
                    row=row_number,
                    column=response_column,
                ).value

                if response is not None and str(response).strip():
                    saved[provider] += 1
                else:
                    missing[provider] += 1

    return saved, missing


def collect_responses(
    sheets_to_run,
    max_new_successes=0,
):
    workbook = load_workbook(
        LOCAL_WORKING_FILE,
        data_only=False,
        keep_links=True,
    )

    restored = restore_checkpoint(workbook)

    if restored:
        save_workbook_to_drive(workbook)

        print(
            f"Restored {restored} successful responses "
            "from the persistent checkpoint."
        )

    successful_calls = 0
    failed_provider_rows = 0
    successes_since_save = 0
    stop_requested = False

    for sheet_name in sheets_to_run:
        worksheet = workbook[sheet_name]

        print(f"\n{'=' * 78}")
        print(sheet_name)
        print("=" * 78)

        for row_number in range(
            2,
            worksheet.max_row + 1,
        ):
            prompt = worksheet.cell(
                row=row_number,
                column=PROMPT_COLUMN,
            ).value

            if prompt is None or not str(prompt).strip():
                continue

            prompt = str(prompt).strip()
            prompt_id = worksheet.cell(
                row=row_number,
                column=PROMPT_ID_COLUMN,
            ).value

            for provider in (
                "ChatGPT",
                "Claude",
                "Gemini",
            ):
                response_column = PROVIDER_COLUMNS[
                    provider
                ]["response_text"]

                existing = worksheet.cell(
                    row=row_number,
                    column=response_column,
                ).value

                if existing is not None and str(existing).strip():
                    continue

                if (
                    max_new_successes > 0
                    and successful_calls >= max_new_successes
                ):
                    stop_requested = True
                    break

                if provider in PROVIDER_DISABLED_REASON:
                    continue

                print(
                    f"{sheet_name} row {row_number} "
                    f"({prompt_id}) -> {provider}"
                )

                try:
                    result, failed_attempts = (
                        call_with_fallback(
                            provider,
                            prompt,
                        )
                    )

                    record = {
                        "sheet": sheet_name,
                        "row": row_number,
                        "prompt_id": prompt_id,
                        "provider": provider,
                        "result": result,
                    }

                    # Save the paid response before relying on the workbook save.
                    append_checkpoint(record)

                    write_result(
                        worksheet,
                        row_number,
                        provider,
                        result,
                    )

                    failed_models = unique_in_order(
                        attempt["model"]
                        for attempt in failed_attempts
                    )

                    if failed_models:
                        append_note(
                            worksheet,
                            row_number,
                            (
                                f"[{result['timestamp']}] "
                                f"{provider} used {result['model']} "
                                "after failures from: "
                                f"{', '.join(failed_models)}."
                            ),
                        )

                    update_completion_status(
                        worksheet,
                        row_number,
                    )

                    successful_calls += 1
                    successes_since_save += 1

                    print(
                        f"    SAVED using {result['model']} "
                        f"({successful_calls} new responses this run)"
                    )

                    if (
                        successes_since_save
                        >= SAVE_WORKBOOK_EVERY_SUCCESSES
                    ):
                        save_workbook_to_drive(workbook)
                        successes_since_save = 0

                        print(
                            "    Workbook progress copied to Google Drive."
                        )

                    time.sleep(
                        PAUSE_AFTER_SUCCESS_SECONDS
                    )

                except KeyboardInterrupt:
                    update_completion_status(
                        worksheet,
                        row_number,
                    )

                    save_workbook_to_drive(workbook)

                    print(
                        "\nStopped manually. "
                        "All successful responses were preserved."
                    )
                    raise

                except ProviderDisabledError as exc:
                    failed_provider_rows += 1

                    append_note(
                        worksheet,
                        row_number,
                        (
                            f"[{utc_timestamp()}] "
                            f"{provider} ACCOUNT/QUOTA ERROR: {exc}"
                        ),
                    )

                    print(
                        f"    {provider} disabled for this run: "
                        f"{exc}"
                    )

                except Exception as exc:
                    failed_provider_rows += 1

                    append_note(
                        worksheet,
                        row_number,
                        (
                            f"[{utc_timestamp()}] "
                            f"{provider} ERROR: "
                            f"{type(exc).__name__}: "
                            f"{str(exc)[:1000]}"
                        ),
                    )

                    print(f"    ERROR: {exc}")

            update_completion_status(
                worksheet,
                row_number,
            )

            if stop_requested:
                break

        if stop_requested:
            break

    save_workbook_to_drive(workbook)

    saved, missing = calculate_progress(workbook)
    workbook.close()

    print(f"\n{'=' * 78}")
    print("RUN FINISHED")
    print("=" * 78)
    print(f"New successful responses: {successful_calls}")
    print(f"Provider-row failures: {failed_provider_rows}")
    print(f"OpenAI saved: {saved['ChatGPT']} / 1000")
    print(f"Claude saved: {saved['Claude']} / 1000")
    print(f"Gemini saved: {saved['Gemini']} / 1000")
    print(
        "Total responses still missing: "
        f"{missing['ChatGPT'] + missing['Claude'] + missing['Gemini']}"
    )
    print(f"Output workbook: {OUTPUT_FILE}")



## 7. One-row test

This makes up to three successful response calls for the first Technology
prompt. These responses are part of the final dataset and will not be repeated
during the full run.


In [ ]:

collect_responses(
    sheets_to_run=["Technology"],
    max_new_successes=3,
)


## 8. Inspect the test responses

In [ ]:

test_workbook = load_workbook(
    LOCAL_WORKING_FILE,
    read_only=True,
    data_only=False,
)

test_sheet = test_workbook["Technology"]

print("Prompt ID:", test_sheet["A2"].value)
print("Prompt:", test_sheet["G2"].value)
print("Completion status:", test_sheet["AY2"].value)

print("\nOPENAI")
print("Model used:", test_sheet["L2"].value)
print(str(test_sheet["Q2"].value or "")[:1200])

print("\nCLAUDE")
print("Model used:", test_sheet["Y2"].value)
print(str(test_sheet["AD2"].value or "")[:1200])

print("\nGEMINI")
print("Model used:", test_sheet["AL2"].value)
print(str(test_sheet["AQ2"].value or "")[:1200])

print("\nCollection notes:")
print(test_sheet["BB2"].value)

test_workbook.close()



## 9. Full collection

The notebook will collect every remaining response. A complete run involves
up to 3,000 provider responses and may take several hours depending on rate
limits and response lengths.

The output and checkpoint are continuously saved in Google Drive. If Colab
disconnects, reconnect, run all cells again and restart this cell.


In [ ]:

RUN_FULL_COLLECTION = True  #@param {type:"boolean"}

if RUN_FULL_COLLECTION:
    collect_responses(
        sheets_to_run=DOMAIN_SHEETS,
        max_new_successes=0,
    )
else:
    print(
        "Full collection has not started. "
        "Set RUN_FULL_COLLECTION to True and run this cell again."
    )


## 10. Check progress without making API calls

In [ ]:

progress_workbook = load_workbook(
    LOCAL_WORKING_FILE,
    read_only=True,
    data_only=False,
)

saved, missing = calculate_progress(
    progress_workbook
)

progress_workbook.close()

print(f"OpenAI saved: {saved['ChatGPT']} / 1000")
print(f"Claude saved: {saved['Claude']} / 1000")
print(f"Gemini saved: {saved['Gemini']} / 1000")
print(
    "Total responses still missing: "
    f"{missing['ChatGPT'] + missing['Claude'] + missing['Gemini']}"
)

print("\nModels used in this Colab session:")

if MODEL_USAGE:
    for key, count in MODEL_USAGE.most_common():
        print(f"  {key}: {count}")
else:
    print("  No new responses have been generated in this session.")


## 11. Download the current completed workbook

In [ ]:

# Ensure the browser download uses the latest Drive copy.
shutil.copy2(
    OUTPUT_FILE,
    LOCAL_WORKING_FILE,
)

print(f"Permanent Google Drive copy:\n{OUTPUT_FILE}")
files.download(str(LOCAL_WORKING_FILE))
